# Credit Risk Default Prediction

This notebook summarizes an end-to-end classification study using a reproducible **synthetic** dataset. The source scripts in `src/` remain the authoritative implementation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'credit_risk_data.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'credit_risk_data.csv'
RESULT_DIR = PROJECT_ROOT / 'results'
IMAGE_DIR = PROJECT_ROOT / 'images'

data = pd.read_csv(DATA_PATH)
data.head()

## Dataset validation

The dataset contains eight financial predictors and the binary target `default`.

In [ ]:
validation = {
    'rows': len(data),
    'predictors': data.shape[1] - 1,
    'missing_values': int(data.isna().sum().sum()),
    'default_rate': f"{data['default'].mean():.2%}",
}
validation

## Exploratory analysis

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'class_balance.png'), width=600))
display(Image(filename=str(IMAGE_DIR / 'default_rate_by_late_payments.png'), width=750))

The class imbalance makes accuracy insufficient by itself. Late-payment count has a strong monotonic relationship with observed default rate.

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'correlation_heatmap.png'), width=850))

## Held-out model comparison

Models were trained on the same stratified 80% training split and evaluated on the untouched 20% test split.

In [ ]:
comparison = pd.read_csv(RESULT_DIR / 'model_comparison.csv', index_col='model')
comparison.style.format('{:.3f}')

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'model_roc_comparison.png'), width=650))
display(Image(filename=str(IMAGE_DIR / 'model_metric_comparison.png'), width=800))

Logistic regression achieved the higher ROC-AUC (0.854), while the class-balanced random forest achieved higher recall at the default 0.50 threshold.

## Model interpretation

In [ ]:
importance = pd.read_csv(RESULT_DIR / 'permutation_importance.csv')
importance.sort_values(['model', 'importance_mean'], ascending=[True, False]).groupby('model').head(5)

In [ ]:
display(Image(filename=str(IMAGE_DIR / 'permutation_importance_comparison.png'), width=850))

Late payments, credit utilization, and credit-history length are the three leading predictors for both models under held-out permutation importance.

## Business-aware thresholds

Thresholds were selected using out-of-fold training predictions under an illustrative assumption that a missed default costs five times as much as a false alert.

In [ ]:
with (RESULT_DIR / 'threshold_metrics.json').open() as file:
    threshold_results = json.load(file)
pd.DataFrame(threshold_results['test_results']).T

## Conclusion

Logistic regression is the preferred model for this synthetic study because it produced the higher ROC-AUC, stronger average precision, lower illustrative business cost, and easier interpretation. These results do not establish real-world lending performance.